# 🤖 AutoML Pilot - Pro Training Template
This notebook allows you to run advanced AutoML on your dataset, generate a full EDA report, and automatically email the results.

In [ ]:
# @title 1. Install Dependencies
!pip install -q pycaret[full] ydata-profiling pandas
print('✅ Dependencies installed!')

In [ ]:
# @title 2. Load Dataset
import pandas as pd
import os
from google.colab import files

uploaded = files.upload()
if uploaded:
    file_name = list(uploaded.keys())[0]
    df = pd.read_csv(file_name)
    print(f'✅ Loaded {file_name}: {df.shape[0]} rows x {df.shape[1]} columns')
    display(df.head())
else:
    print('❌ No file uploaded.')

In [ ]:
# @title 3. Automated EDA
from ydata_profiling import ProfileReport

run_eda = True # @param {type:"boolean"}
if run_eda:
    print('Generating EDA Report...')
    profile = ProfileReport(df, explorative=True, minimal=True)
    profile.to_file("eda_report.html")
    print('✅ EDA Report saved as eda_report.html')

In [ ]:
# @title 4. AutoML Training
from pycaret.classification import setup as cls_setup, compare_models as cls_compare, pull as cls_pull, save_model as cls_save, finalize_model as cls_finalize
from pycaret.regression import setup as reg_setup, compare_models as reg_compare, pull as reg_pull, save_model as reg_save, finalize_model as reg_finalize

target_column = 'target' # @param {type:"string"}
task_type = 'classification' # @param ["classification", "regression"]

print(f'🚀 Starting AutoML for {task_type}...')

if task_type == 'classification':
    s = cls_setup(data=df, target=target_column, session_id=123, verbose=False)
    best_model = cls_compare()
    leaderboard = cls_pull()
    final_model = cls_finalize(best_model)
    cls_save(final_model, 'best_model')
else:
    s = reg_setup(data=df, target=target_column, session_id=123, verbose=False)
    best_model = reg_compare()
    leaderboard = reg_pull()
    final_model = reg_finalize(best_model)
    reg_save(final_model, 'best_model')

print('✅ Training complete! Best model saved.')
display(leaderboard.head())

In [ ]:
# @title 5. Email Results
import smtplib, ssl, json
from email.mime.text import MIMEText
from email.mime.multipart import MIMEMultipart

recipient_email = '' # @param {type:"string"}
sender_email = '' # @param {type:"string"}
sender_password = '' # @param {type:"string"}

if recipient_email and sender_email and sender_password:
    print(f'Sending results to {recipient_email}...')
    try:
        msg = MIMEMultipart("alternative")
        msg["Subject"] = f"AutoML Colab Results - {target_column}"
        msg["From"] = sender_email
        msg["To"] = recipient_email
        
        metrics_html = leaderboard.head(5).to_html()
        html_body = f"""
        <html>
        <body>
            <h2>🚀 AutoML Pilot - Colab Training Report</h2>
            <p>Target: <b>{target_column}</b> | Task: <b>{task_type}</b></p>
            <h3>📊 Top Models</h3>
            {metrics_html}
            <p>Model file 'best_model.pkl' is ready for download in Colab.</p>
        </body>
        </html>
        """
        msg.attach(MIMEText(html_body, "html"))
        context = ssl.create_default_context()
        with smtplib.SMTP_SSL("smtp.gmail.com", 465, context=context) as server:
            server.login(sender_email, sender_password)
            server.sendmail(sender_email, recipient_email, msg.as_string())
        print('✅ Email sent successfully!')
    except Exception as e:
        print(f'❌ Email failed: {str(e)}')
else:
    print('ℹ️ Email config incomplete. Skipping email.')

In [ ]:
# @title 6. Download Model
from google.colab import files
if os.path.exists('best_model.pkl'):
    files.download('best_model.pkl')
else:
    print('❌ Model file not found.')